In [1]:
import pandas as pd

# Load the full dataset
df = pd.read_csv('/content/emails.csv.zip')

# Take user input for number of test emails
num_test_emails = int(input("Enter the number of test emails to use from file: "))

# Ensure number is reasonable
num_test_emails = min(num_test_emails, len(df))

# Slice the test dataset part to use (e.g. last part of dataset)
test_emails = df.tail(num_test_emails)

print(f"Using {num_test_emails} emails from dataset as test data:")

print(test_emails[['Category', 'Message']].head())
# Step 3: Convert Labels to Numeric
df.loc[df['Category'] == 'spam', 'Category'] = 0
df.loc[df['Category'] == 'ham',  'Category'] = 1

# Step 4: Split Messages and Labels
X = df['Message']
y = df['Category'].astype(int)

# Step 5: Train-Test Split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=3)
print('Train size:', len(X_train), 'Test size:', len(X_test))

# Step 6: Text Feature Extraction (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(min_df=1, stop_words='english', lowercase=True)
X_train_features = vectorizer.fit_transform(X_train)
X_test_features = vectorizer.transform(X_test)

# Step 7: Train Logistic Regression
from sklearn.linear_model import LogisticRegression
model = LogisticRegression()
model.fit(X_train_features, y_train)

# Step 8: Evaluate Accuracy
from sklearn.metrics import accuracy_score
y_train_pred = model.predict(X_train_features)
y_test_pred = model.predict(X_test_features) # Corrected this line
print('Accuracy on training data:', accuracy_score(y_train, y_train_pred))
print('Accuracy on test data:', accuracy_score(y_test, y_test_pred)) # Corrected this line

# Step 9: Predict New Email
def predict_spam(email_message):
    features = vectorizer.transform([email_message])
    result = model.predict(features)[0]
    return 'Spam' if result == 0 else 'Ham'

# Example Usage:
print(predict_spam("Congratulations! You've won a prize. Click here to claim."))
print(predict_spam("Can we meet tomorrow regarding our project update?"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/emails.csv.zip'

In [ ]:
import zipfile
import os
import pandas as pd
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

def clean_email_text(text):
    text = re.sub(r'<.*?>', '', text)  # Remove HTML tags
    text = re.sub(r'[^a-zA-Z ]', ' ', text)  # Remove numbers/punct
    text = text.lower()  # Lowercase
    tokens = text.split()
    tokens = [word for word in tokens if word not in set(stopwords.words('english'))]
    ps = PorterStemmer()
    stemmed = [ps.stem(word) for word in tokens]
    return ' '.join(stemmed)

# 1. User input for path
zip_path = input("/content/emails.csv.zip ").strip()
extract_dir = os.path.dirname(zip_path)
print("[Step 1] Got user input ⇒", zip_path)

# 2. Extract zip
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print(f"[Step 2] Extracted zip to {extract_dir}")

# 3. Load CSV
csv_path = ''
for f in os.listdir(extract_dir):
    if f.endswith('.csv'):
        csv_path = os.path.join(extract_dir, f)
        break
if not csv_path:
    print("CSV not found after extracting. Exiting.")
    exit()
data = pd.read_csv(csv_path)
print(f"[Step 3] Loaded data ⇒ {len(data)} emails")

# 4. Preprocess
print("[Step 4] Cleaning email texts (this may take a moment)...")
data['clean_body'] = data['text_body'].astype(str).apply(clean_email_text)
print("[Step 4] Texts cleaned.")

# 5. Features: TF-IDF
vectorizer = TfidfVectorizer(max_features=1000)
X = vectorizer.fit_transform(data['clean_body'])
print("[Step 5] Extracted TF-IDF features.")

y = data['label'].map({'ham': 0, 'spam': 1})
if y.isnull().any():
    print("[Warning] Some emails have missing/unknown labels. Check your dataset.")

# 6. Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"[Step 6] Data split: {X_train.shape[0]} train, {X_test.shape[0]} test emails.")

# 7. Model: Naive Bayes
clf = MultinomialNB()
clf.fit(X_train, y_train)
print("[Step 7] Trained Multinomial Naive Bayes classifier.")

# 8. Evaluation
y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
conf_mat = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred)

print("[Step 8] Evaluation results:")
print(f"  - Accuracy:  {accuracy:.2f}")
print(f"  - Precision: {precision:.2f}")
print(f"  - Recall:    {recall:.2f}")
print(f"  - F1-score:  {f1:.2f}")
print("\nConfusion Matrix:")
print(conf_mat)

# Optionally save results
results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1-score'],
    'Score': [accuracy, precision, recall, f1]
})
results_fp = os.path.join(extract_dir, 'spam_detection_results.csv')
confmat_fp = os.path.join(extract_dir, 'spam_confusion_matrix.csv')
report_fp = os.path.join(extract_dir, 'spam_class_report.txt')
results.to_csv(results_fp, index=False)
pd.DataFrame(conf_mat, columns=['Pred Ham', 'Pred Spam'], index=['Actual Ham', 'Actual Spam']).to_csv(confmat_fp)
with open(report_fp, 'w') as f:
    f.write(report)
print(f"[Step 9] Results saved to:\n - {results_fp}\n - {confmat_fp}\n - {report_fp}")


/content/emails.csv.zip https://colab.research.google.com/drive/1wrlp5omOd5O0D_jXhguQAPjA9AJPytYd#
[Step 1] Got user input ⇒ https://colab.research.google.com/drive/1wrlp5omOd5O0D_jXhguQAPjA9AJPytYd#


FileNotFoundError: [Errno 2] No such file or directory: 'https://colab.research.google.com/drive/1wrlp5omOd5O0D_jXhguQAPjA9AJPytYd#'